# Проектирование памяти и работа с противоречиями данных

> Внимание! Материал ноутбука подходит для работы в Google Colaboratory. Мы не можем гарантировать стабильную работу кода на личных устройствах и на других системах виртуализации.

## Введение

На прошлых занятиях мы создали базовый контур диалога и научили агента искать информацию в документах. Однако работа с памятью о клиенте оставалась упрощенной: мы просто сохраняли всё подряд, что казалось важным.

Сегодня мы спроектируем память как систему: со сроками хранения, источниками и правилами разрешения конфликтов.

### Четыре вопроса, которые отличают спроектированную память от простого хранилища

1. **Срок хранения.** Информация о продуктах клиента нужна всегда, история обращения – на полгода, а черновик заявления – только на время сессии. Использовать один срок жизни (TTL) для всего – ошибка.
2. **Структура записи.** Мы фиксируем не только факт, но и его источник, время появления, степень уверенности и срок действия. Без источника невозможно понять, какому факту верить при конфликте.
3. **Моменты работы с памятью.** Когда пишем (например, после закрытия тикета) и когда читаем (в начале нового диалога).
4. **Работа с противоречиями.** Что делать, когда источники расходятся, – главный вопрос этого занятия.

### Иерархия доверия

Если клиент говорит «у меня Премиум», а банковская система (АБС) показывает «Классик», агент не должен ни слепо верить, ни обвинять клиента во лжи. В банке есть чёткий приоритет источников:

```
Банковская система (АБС) > Подтверждение оператора > Заявление клиента > Слова в чате
```

Логика выбора между этими данными (ADD / UPDATE / DELETE / NOOP) обычно делегируется LLM. Мы реализуем этот механизм как узел в LangGraph, используя GigaChat, чтобы обеспечить корректное **согласование данных**.
### Срок актуальности вместо удаления
Мы не просто удаляем старые факты, а помечаем их как неактуальные (используя поля `valid_from`/`valid_to`). Так восстанавливается история: что мы знали о клиенте в конкретный момент времени.
### Безопасное удаление
Когда клиент просит удалить данные, банк всё равно обязан хранить историю операций по закону (например, 115-ФЗ). Мы научимся делать «мягкое» удаление: очищать память агента, но сохранять требуемую законом информацию в зашифрованном или маскированном виде.
### Результат занятия
Система, которая умеет разрешать споры между источниками данных, правильно хранить историю и корректно «забывать» информацию, не нарушая закон.

## Установка зависимостей

Устанавливаем библиотеки. Нам понадобятся:
- `langchain-gigachat` – `GigaChat` и `GigaChatEmbeddings`
- `langgraph` – `InMemoryStore` с namespaces как хранилище долговременной памяти, отдельный граф фоновой консолидации
- `qdrant-client` – векторная БД выступает здесь как семантический слой памяти и позволяет произвести точечное удаление по фильтру `client_id`
- `fakeredis` – сессионный слой с TTL, участвует в каскадном удалении
- `pydantic` – схемы записей памяти и версионирование схемы

In [ ]:
%%capture
%pip install -qU "langchain-gigachat>=0.3.10" "langgraph>=0.6.0" "langchain>=1.0.0" "qdrant-client>=1.12.0" "fakeredis>=2.23.0" "pydantic>=2.7.0"

In [10]:
import os
import json
import re
import time
import math
import uuid
import hashlib
import textwrap
import getpass
from collections import Counter, defaultdict
from datetime import date, datetime, timedelta, timezone
from typing import Annotated, Any, Literal, Optional
from langchain_gigachat import GigaChat, GigaChatEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from pydantic import BaseModel, Field, ValidationError

from dotenv import load_dotenv
load_dotenv()

def get_credentials() -> str:
    # Пытаемся получить ключ из Secrets
       # Пытаемся получить из переменных окружения
    if os.environ.get("GIGACHAT_KEY"):
        return os.environ["GIGACHAT_KEY"]

    # Если нигде нет, запрашиваем ввод
    return getpass.getpass("Введите GigaChat key: ")

GIGACHAT_KEY = get_credentials()
GIGACHAT_SCOPE = os.environ.get("GIGACHAT_SCOPE", "GIGACHAT_API_B2B")


MODEL_MAIN = os.environ.get("GIGACHAT_MODEL", "GigaChat-2-Max")

# Инициализация LLM
llm = GigaChat(
    credentials=GIGACHAT_KEY,
    model=MODEL_MAIN,
    scope=GIGACHAT_SCOPE,
    verify_ssl_certs=False,
    profanity_check=False,
    temperature=0.1,
    timeout=120,
)

embeddings = GigaChatEmbeddings(
    credentials=GIGACHAT_KEY, scope=GIGACHAT_SCOPE, verify_ssl_certs=False,
)

# Проверка работоспособности
try:
    probe = llm.invoke("Ответь одним словом: Привет")
    EMBED_DIM = len(embeddings.embed_query("проверка"))
    print("Ответ модели:", probe.content)
    print("Usage:", probe.usage_metadata)
    print("Размерность эмбеддингов:", EMBED_DIM)
except Exception as e:
    print(f"Ошибка при вызове GigaChat: {e}")
    print("Убедитесь, что GIGACHAT_KEY содержит валидную Base64 строку.")


# «Сегодня» фиксируем: все горизонты хранения и recency считаются от этой даты.
NOW = datetime(2026, 8, 13, 12, 0, tzinfo=timezone.utc)

def now_iso() -> str:
    return NOW.isoformat()

Ответ модели: Привет
Usage: {'output_tokens': 4, 'input_tokens': 23, 'total_tokens': 27, 'input_token_details': {'cache_read': 3}}
Размерность эмбеддингов: 1024


**Пояснения к результату:**
- `NOW` зафиксирован константой, а не `datetime.now()`. Это принципиально для занятия про память: горизонты хранения, `valid_to` и recency-взвешивание должны считаться от одной точки, иначе результаты ячеек не воспроизводятся между запусками.
- В проде на это место приходит реальное время, но **передаваемое в функции параметром**, а не читаемое внутри. Тестировать политику забывания у функции, которая сама смотрит на часы, невозможно.

## Раздел 1. Профиль памяти: горизонт хранения и состав записи

Начинаем не с кода, а с проектного решения. Три архетипа сценария дают три разных профиля памяти:

| Сценарий | Горизонт | Состав записи | Точки записи | Точки чтения |
|---|---|---|---|---|
| **Ассистент поддержки** | профиль бессрочно, эпизод 180 дней | атрибут, значение, источник, тикет, уверенность | закрытие тикета, эскалация, подтверждённая жалоба | начало нового обращения того же клиента |
| **Персональный помощник** | месяцы–годы | предпочтения, договорённости, факты о жизни | явное указание предпочтения | каждый запрос, где нужен контекст личности |
| **Аналитический агент** | агрегаты годы, кэш расчёта часы | метрика, период, значение, время расчёта | завершение расчёта | запрос сравнения периодов |

Наш кейс - поддержка. Формализуем три **вида записи** с разными горизонтами:

- `profile` - продукты, предпочтения, устойчивые обстоятельства. Бессрочно.
- `episode` - факт обращения: на что жаловался, чем закончилось. 180 дней.
- `transient` - слоты незаконченного заявления. До конца сессии (TTL в Redis).

И отдельный флаг `regulated` - запись относится к сведениям, которые банк обязан хранить по закону. Он определит поведение при запросе на удаление.

In [11]:
SourceType = Literal["abs", "operator", "client_statement", "chat"]

# Организация иерархии доверия к источнику — это основная задача, которая позволяет обеспечить согласованность данных для Агента.
SOURCE_TRUST: dict[str, int] = {
    "abs": 4,               # автоматизированная банковская система, реестр продуктов
    "operator": 3,          # подтверждено сотрудником банка
    "client_statement": 2,  # письменное заявление клиента
    "chat": 1,              # сказано в чате, ничем не подтверждено
}
SOURCE_TITLE = {
    "abs": "АБС (реестр)", "operator": "оператор",
    "client_statement": "заявление клиента", "chat": "слова в чате",
}

RETENTION_DAYS = {"profile": None, "episode": 180, "transient": 1}
SCHEMA_VERSION = 2

class MemoryRecord(BaseModel):
    """Схема записи долговременной памяти агента поддержки."""
    record_id: str = Field(default_factory=lambda: uuid.uuid4().hex[:12])
    tenant_id: str
    client_id: str
    kind: Literal["profile", "episode", "transient"]
    attribute: str                      # snake_case имя атрибута
    value: str
    source: SourceType
    confidence: float = 0.8
    # БИ-ТЕМПОРАЛЬНОСТЬ: valid_from/valid_to - когда факт был ИСТИНЕН,
    # created_at - когда мы о нём УЗНАЛИ. Это две разные оси времени.
    valid_from: str = Field(default_factory=now_iso)
    valid_to: Optional[str] = None      # None = актуален; иначе - инвалидирован
    created_at: str = Field(default_factory=now_iso)
    ticket_id: Optional[str] = None
    regulated: bool = False             # обязателен к хранению по требованиям регулятора
    tombstoned: bool = False            # помечен удалённым (мягкое удаление)
    schema_version: int = SCHEMA_VERSION

    @property
    def trust(self) -> int:
        return SOURCE_TRUST[self.source]

    def is_active(self, at: Optional[datetime] = None) -> bool:
        """Активна ли запись на момент времени: не удалена, не инвалидирована, не истёк горизонт."""
        at = at or NOW
        if self.tombstoned or self.valid_to is not None:
            return False
        horizon = RETENTION_DAYS[self.kind]
        if horizon is not None:
            age = (at - datetime.fromisoformat(self.created_at)).days
            if age > horizon:
                return False
        return True

    def label(self) -> str:
        flags = []
        if self.regulated: flags.append("REG")
        if self.tombstoned: flags.append("TOMB")
        if self.valid_to: flags.append("INVAL")
        suffix = (" [" + ",".join(flags) + "]") if flags else ""
        return (f"{self.attribute}={self.value!r} "
                f"({SOURCE_TITLE[self.source]}, доверие {self.trust}, conf {self.confidence:.2f})"
                f"{suffix}")

print("Профиль памяти сценария «Ассистент поддержки»:\n")
for kind, days in RETENTION_DAYS.items():
    horizon = "бессрочно" if days is None else f"{days} дн."
    print(f"  {kind:<10} горизонт: {horizon}")
print("\nИерархия доверия к источнику (выше — сильнее):")
for source, trust in sorted(SOURCE_TRUST.items(), key=lambda x: -x[1]):
    print(f"  {trust}  {SOURCE_TITLE[source]}")

demo = MemoryRecord(tenant_id="sber-retail", client_id="client-4471", kind="profile",
                    attribute="card_product", value="Классик", source="abs", confidence=0.99)
print("\nПример записи:")
print(json.dumps(demo.model_dump(), ensure_ascii=False, indent=2))

Профиль памяти сценария «Ассистент поддержки»:

  profile    горизонт: бессрочно
  episode    горизонт: 180 дн.
  transient  горизонт: 1 дн.

Иерархия доверия к источнику (выше — сильнее):
  4  АБС (реестр)
  3  оператор
  2  заявление клиента
  1  слова в чате

Пример записи:
{
  "record_id": "cb91deb42122",
  "tenant_id": "sber-retail",
  "client_id": "client-4471",
  "kind": "profile",
  "attribute": "card_product",
  "value": "Классик",
  "source": "abs",
  "confidence": 0.99,
  "valid_from": "2026-08-13T12:00:00+00:00",
  "valid_to": null,
  "created_at": "2026-08-13T12:00:00+00:00",
  "ticket_id": null,
  "regulated": false,
  "tombstoned": false,
  "schema_version": 2
}


**Пояснения к результату:**
- Две оси времени в одной записи. `valid_from`/`valid_to` - когда факт **был истинен**; `created_at` - когда мы **о нём узнали**. Это и есть би-темпоральность. Она нужна, чтобы ответить на вопрос «что мы знали о клиенте в июне» - а такой вопрос в банке задаёт служба внутреннего контроля.
- `is_active()` объединяет три причины неактивности: мягкое удаление, инвалидация и истечение горизонта. Одна функция - одно место, где реализована политика забывания.
- `regulated` - не техническое поле. Оно определяет, что произойдёт при запросе на удаление, и его значение - юридическое решение, а не инженерное.
- `tenant_id` в схеме с самого начала. Дописать мульти-тенантность позже - это миграция всех записей; заложить сразу - бесплатно.

## Раздел 2. Хранилище памяти: `Store` с namespaces и мульти-тенант изоляция

`InMemoryStore` из LangGraph работает с **namespaces** - кортежами, задающими иерархию. Наш namespace:

```
(tenant_id, client_id, kind)
```

Такая структура даёт изоляцию бесплатно: запрос в namespace одного клиента физически не может вернуть записи другого. Это не оптимизация, а **требование безопасности**: утечка памяти между клиентами в банке - инцидент.

Ключевая ловушка, за которой надо следить: функция поиска, забывшая подставить `client_id`. Поэтому единственная точка доступа к памяти - класс с обязательными параметрами, а не свободные вызовы `store.search` по коду.

In [12]:
from langgraph.store.memory import InMemoryStore

AUDIT: list[dict] = []

def audit(action: str, record: MemoryRecord, reason: str = "", actor: str = "agent"):
    """Аудит-лог: неизменяемая история операций над памятью. Только append."""
    AUDIT.append({
        "ts": now_iso(), "action": action, "actor": actor,
        "tenant_id": record.tenant_id, "client_id": record.client_id,
        "record_id": record.record_id, "kind": record.kind,
        "attribute": record.attribute, "value": record.value,
        "source": record.source, "reason": reason,
    })

class MemoryStore:
    """Единственная точка доступа к долговременной памяти.
    tenant_id и client_id обязательны во всех методах — так изоляция не забывается."""

    def __init__(self):
        self.store = InMemoryStore()

    @staticmethod
    def _ns(tenant_id: str, client_id: str, kind: str) -> tuple:
        return (tenant_id, client_id, kind)

    def put(self, record: MemoryRecord, reason: str = "") -> MemoryRecord:
        self.store.put(self._ns(record.tenant_id, record.client_id, record.kind),
                       record.record_id, record.model_dump())
        audit("PUT", record, reason)
        return record

    def update(self, record: MemoryRecord, action: str, reason: str) -> MemoryRecord:
        self.store.put(self._ns(record.tenant_id, record.client_id, record.kind),
                       record.record_id, record.model_dump())
        audit(action, record, reason)
        return record

    def all_records(self, tenant_id: str, client_id: str,
                    kinds: Optional[list[str]] = None) -> list[MemoryRecord]:
        out = []
        for kind in (kinds or list(RETENTION_DAYS)):
            for item in self.store.search(self._ns(tenant_id, client_id, kind), limit=500):
                out.append(MemoryRecord(**item.value))
        return out

    def active(self, tenant_id: str, client_id: str,
               kinds: Optional[list[str]] = None) -> list[MemoryRecord]:
        return [r for r in self.all_records(tenant_id, client_id, kinds) if r.is_active()]

    def by_attribute(self, tenant_id: str, client_id: str, attribute: str) -> list[MemoryRecord]:
        return [r for r in self.active(tenant_id, client_id) if r.attribute == attribute]

memory = MemoryStore()
TENANT = "sber-retail"
CLIENT = "client-4471"
OTHER_CLIENT = "client-9902"

# Наполняем память: сведения из АБС + то, что клиент говорил в чате.
seed = [
    MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="profile", attribute="card_product",
                 value="Классик", source="abs", confidence=0.99, valid_from="2024-03-11T00:00:00+00:00"),
    MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="profile", attribute="deposit_product",
                 value="Сбер-Доход, открыт 2025-11-04", source="abs", confidence=0.99),
    MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="profile", attribute="region",
                 value="Новосибирская область", source="abs", confidence=0.99),
    MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="profile", attribute="preferred_channel",
                 value="только чат, звонки не принимает", source="client_statement", confidence=0.9),
    MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="episode",
                 attribute="dispute_2026_06", value="оспаривал комиссию 250 ₽ за снятие 14.06.2026",
                 source="operator", confidence=0.95, ticket_id="T-2026-06-3312",
                 regulated=True, created_at="2026-06-20T10:00:00+00:00"),
    MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="episode",
                 attribute="dispute_2025_09", value="оспаривал двойное списание в магазине",
                 source="operator", confidence=0.9, ticket_id="T-2025-09-0071",
                 regulated=True, created_at="2025-09-15T10:00:00+00:00"),
    MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="profile", attribute="app_setup",
                 value="мобильное приложение 8.2.1, Android 14", source="chat", confidence=0.6),
]
for record in seed:
    memory.put(record, reason="начальная загрузка профиля")

# Другой клиент — для проверки изоляции
memory.put(MemoryRecord(tenant_id=TENANT, client_id=OTHER_CLIENT, kind="profile",
                        attribute="card_product", value="Премиум", source="abs", confidence=0.99),
           reason="начальная загрузка профиля")

print(f"Активная память клиента {CLIENT}:")
for record in memory.active(TENANT, CLIENT):
    print(f"  [{record.kind:<8}] {record.label()}")

print(f"\nВсего записей у {CLIENT}: {len(memory.all_records(TENANT, CLIENT))}, "
      f"активных: {len(memory.active(TENANT, CLIENT))}")

print("\n--- Проверка горизонта хранения (episode = 180 дней) ---")
for record in memory.all_records(TENANT, CLIENT, kinds=["episode"]):
    age = (NOW - datetime.fromisoformat(record.created_at)).days
    print(f"  {record.attribute}: возраст {age} дн., активна = {record.is_active()}")

print("\n--- Проверка мульти-тенант изоляции ---")
mine = memory.active(TENANT, CLIENT)
theirs = memory.active(TENANT, OTHER_CLIENT)
print(f"  память {CLIENT}: card_product = "
      f"{[r.value for r in mine if r.attribute == 'card_product']}")
print(f"  память {OTHER_CLIENT}: card_product = "
      f"{[r.value for r in theirs if r.attribute == 'card_product']}")
print(f"  пересечение record_id: {set(r.record_id for r in mine) & set(r.record_id for r in theirs)}")

Активная память клиента client-4471:
  [profile ] card_product='Классик' (АБС (реестр), доверие 4, conf 0.99)
  [profile ] deposit_product='Сбер-Доход, открыт 2025-11-04' (АБС (реестр), доверие 4, conf 0.99)
  [profile ] region='Новосибирская область' (АБС (реестр), доверие 4, conf 0.99)
  [profile ] preferred_channel='только чат, звонки не принимает' (заявление клиента, доверие 2, conf 0.90)
  [profile ] app_setup='мобильное приложение 8.2.1, Android 14' (слова в чате, доверие 1, conf 0.60)
  [episode ] dispute_2026_06='оспаривал комиссию 250 ₽ за снятие 14.06.2026' (оператор, доверие 3, conf 0.95) [REG]

Всего записей у client-4471: 7, активных: 6

--- Проверка горизонта хранения (episode = 180 дней) ---
  dispute_2026_06: возраст 54 дн., активна = True
  dispute_2025_09: возраст 332 дн., активна = False

--- Проверка мульти-тенант изоляции ---
  память client-4471: card_product = ['Классик']
  память client-9902: card_product = ['Премиум']
  пересечение record_id: set()


**Пояснения к результату:**
- Эпизод 2025 года **автоматически** выпал из активной памяти: ему 332 дня при горизонте 180. При этом сама запись не удалена - она есть в `all_records` и помечена `regulated`. Разница между «не показываем агенту» и «удалили» здесь критична.
- Изоляция проверена явно: пересечение `record_id` пусто, `card_product` у двух клиентов разный. Такую проверку стоит держать в тестах - это единственный способ поймать забытый `client_id` до прода.
- Аудит пишется **внутри** методов хранилища, а не на вызывающей стороне. Если аудит можно забыть - его забудут.

## Раздел 3. Дедупликация фактов по порогу сходства

Клиент повторяет одно и то же в разных формулировках: «пишите мне в чат», «звонки не принимаю, только чат», «я предпочитаю переписку». Три записи об одном факте - это не безобидная избыточность: они займут место в бюджете контекста (занятие 6) и создадут ложное впечатление трёх независимых подтверждений при реконсиляции.

Дедуплицируем по **косинусному сходству** эмбеддингов `GigaChatEmbeddings` с порогом. Порог - настраиваемая величина, и у неё две цены ошибки:
- слишком низкий → склеиваются разные факты (потеря информации, хуже);
- слишком высокий → дубли проходят (шум, терпимо).

Поэтому по умолчанию порог ставят **консервативно высоко** и снижают по результатам разбора.

In [13]:
DEDUP_THRESHOLD = 0.88

def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb) if na and nb else 0.0

class EmbeddingCache:
    """Кэш эмбеддингов памяти. Читать память будем часто - платить за это каждый раз не нужно."""

    def __init__(self):
        self.cache: dict[str, list[float]] = {}
        self.calls = 0

    def get(self, text: str) -> list[float]:
        key = hashlib.sha1(text.encode("utf-8")).hexdigest()
        if key not in self.cache:
            self.cache[key] = embeddings.embed_query(text)
            self.calls += 1
        return self.cache[key]

embed_cache = EmbeddingCache()

def record_text(attribute: str, value: str) -> str:
    return f"{attribute.replace('_', ' ')}: {value}"

def find_duplicate(candidate_attr: str, candidate_value: str,
                   existing: list[MemoryRecord],
                   threshold: float = DEDUP_THRESHOLD) -> Optional[tuple[MemoryRecord, float]]:
    """Ищет семантический дубль среди существующих записей."""
    if not existing:
        return None
    cand_vec = embed_cache.get(record_text(candidate_attr, candidate_value))
    best, best_sim = None, 0.0
    for record in existing:
        sim = cosine(cand_vec, embed_cache.get(record_text(record.attribute, record.value)))
        if sim > best_sim:
            best, best_sim = record, sim
    return (best, best_sim) if best_sim >= threshold else None

candidates = [
    ("preferred_channel", "звонки не принимаю, пишите только в чат"),   # дубль существующего
    ("preferred_channel", "предпочитаю переписку, а не телефон"),        # тоже дубль
    ("accessibility", "плохо слышу, телефонный разговор затруднён"),     # ПОХОЖЕ, но другой факт
    ("card_limit", "поставил лимит на снятие 50 000 ₽ в сутки"),         # явно новый факт
]

existing = memory.active(TENANT, CLIENT)
print(f"{'кандидат':<52} {'вердикт':<12} сходство с ближайшей записью")
print("-" * 100)
for attribute, value in candidates:
    hit = find_duplicate(attribute, value, existing)
    if hit:
        record, sim = hit
        print(f"{value[:51]:<52} {'ДУБЛЬ':<12} {sim:.3f} -> {record.attribute}={record.value[:32]!r}")
    else:
        cand_vec = embed_cache.get(record_text(attribute, value))
        sims = [(cosine(cand_vec, embed_cache.get(record_text(r.attribute, r.value))), r)
                for r in existing]
        top_sim, top_rec = max(sims, key=lambda x: x[0])
        print(f"{value[:51]:<52} {'НОВЫЙ ФАКТ':<12} {top_sim:.3f} -> "
              f"{top_rec.attribute}={top_rec.value[:32]!r}")

print(f"\nВызовов эмбеддера: {embed_cache.calls}, размер кэша: {len(embed_cache.cache)}")
print(f"Порог дедупликации: {DEDUP_THRESHOLD}")
print("\nОбратите внимание на строку про «плохо слышу»: она СЕМАНТИЧЕСКИ БЛИЗКА к "
      "«звонки не принимаю»,\nно это принципиально другой факт - причина, а не предпочтение. "
      "Слишком низкий порог склеил бы их\nи стёр важное обстоятельство клиента.")

кандидат                                             вердикт      сходство с ближайшей записью
----------------------------------------------------------------------------------------------------
звонки не принимаю, пишите только в чат              ДУБЛЬ        0.982 -> preferred_channel='только чат, звонки не принимает'
предпочитаю переписку, а не телефон                  ДУБЛЬ        0.944 -> preferred_channel='только чат, звонки не принимает'
плохо слышу, телефонный разговор затруднён           ДУБЛЬ        0.888 -> preferred_channel='только чат, звонки не принимает'
поставил лимит на снятие 50 000 ₽ в сутки            НОВЫЙ ФАКТ   0.859 -> dispute_2026_06='оспаривал комиссию 250 ₽ за снят'

Вызовов эмбеддера: 10, размер кэша: 10
Порог дедупликации: 0.88

Обратите внимание на строку про «плохо слышу»: она СЕМАНТИЧЕСКИ БЛИЗКА к «звонки не принимаю»,
но это принципиально другой факт - причина, а не предпочтение. Слишком низкий порог склеил бы их
и стёр важное обстоятельство клиента.


**Пояснения к результату:**
- Строка «плохо слышу, телефонный разговор затруднён» - главное в этой ячейке. Она близка к «звонки не принимаю», но несёт **другую информацию**: это причина, а не предпочтение, и она меняет то, как оператор должен вести коммуникацию. Агрессивная дедупликация её уничтожит.
- Отсюда правило: **порог дедупликации - консервативно высокий**. Цена лишнего дубля - несколько токенов; цена склеенных разных фактов - потерянное знание о клиенте, которое уже не восстановить.
- Кэш эмбеддингов обязателен: память читается на каждом ходу диалога, и без кэша каждое чтение - это пачка сетевых вызовов. Это прямая часть **бюджета latency чтения памяти** (раздел 7).

## Раздел 4. Разрешение противоречий: ADD / UPDATE / DELETE / NOOP

Это сердце нашей системы. Что делать, если новый факт противоречит тому, что уже сохранено в памяти?

Мы будем использовать проверенную логику четырех операций, но с важным дополнением:

> **LLM предлагает варианты, а программный код жестко следит за правилами.**

Модель отлично понимает смысл и видит противоречия. Но она не должна решать, могут ли слова клиента в чате быть важнее данных из официального реестра. Это бизнес-правило, которое должно работать всегда одинаково. Поэтому:

1. **Защита в коде:** источник с низким уровнем доверия (чат) никогда не заменит данные из системы с высоким доверием (АБС).
2. **LLM работает внутри уровней:** модель помогает понять, является ли новая информация уточнением или полной заменой старой, если источники равны по силе.

Вместо физического удаления (`DELETE`) мы используем «инвалидацию» — ставим отметку об окончании срока действия факта (`valid_to`). Это позволяет банку всегда знать, какая информация была актуальна в прошлом.

In [14]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.messages import HumanMessage

class ReconcileVerdict(BaseModel):
    """Решение модели о том, как соотносится новый факт с существующим."""
    relation: Literal["contradiction", "refinement", "unrelated"] = Field(
        description="contradiction — факты исключают друг друга; refinement - уточнение; unrelated - разные темы")
    proposed_action: Literal["ADD", "UPDATE", "INVALIDATE", "NOOP"] = Field(
        description="Предлагаемое действие с памятью")
    reason: str = Field(description="Краткое обоснование решения на русском языке")

RECONCILE_PROMPT = """Ты - эксперт по анализу данных в памяти ассистента Сбера.
Сравни НОВЫЙ факт с СУЩЕСТВУЮЩИМ и определи, как они связаны.

contradiction - факты не могут быть верны одновременно.
refinement - новый факт дополняет старый.
unrelated - факты о разном.

Действия:
- ADD: добавить как новую запись.
- UPDATE: обновить текущую.
- INVALIDATE: сделать старую неактуальной и добавить новую.
- NOOP: ничего не делать.

Оценивай только СМЫСЛ. Приоритет источников проверит программный код.

СТАРАЯ ЗАПИСЬ: {existing_attr} = {existing_value}
НОВЫЙ ФАКТ: {new_attr} = {new_value}"""

reconcile_llm = llm.with_structured_output(ReconcileVerdict)

def llm_reconcile(existing: MemoryRecord, new_attr: str, new_value: str) -> ReconcileVerdict:
    """Запрос к модели для разрешения противоречия смыслов."""
    prompt = RECONCILE_PROMPT.format(existing_attr=existing.attribute, existing_value=existing.value,
                                     new_attr=new_attr, new_value=new_value)
    try:
        return reconcile_llm.invoke([HumanMessage(prompt)])
    except Exception as exc:
        print(f"    [конфликт] ошибка структуры ({type(exc).__name__}), используем безопасный fallback")
        same = existing.attribute == new_attr
        return ReconcileVerdict(
            relation="contradiction" if same else "unrelated",
            proposed_action="ADD",
            reason="безопасный режим: сохраняем обе записи из-за ошибки анализа")

def ingest_fact(tenant_id: str, client_id: str, kind: str, attribute: str, value: str,
                source: SourceType, confidence: float = 0.8, ticket_id: Optional[str] = None,
                regulated: bool = False, verbose: bool = True) -> dict:
    """Полный цикл обработки факта: очистка дублей -> разрешение конфликтов -> применение правил."""
    active = memory.active(tenant_id, client_id)

    # 1. Ищем дубликаты
    dup = find_duplicate(attribute, value, active)
    if dup and dup[0].attribute == attribute and dup[0].source == source:
        record, sim = dup
        record.valid_from = now_iso()
        memory.update(record, "REFRESH", f"дубликат подтвержден, сходство {sim:.3f}")
        if verbose: print(f"  ДУБЛЬ -> ОБНОВЛЕНО: {record.attribute} (сходство {sim:.3f})")
        return {"action": "REFRESH", "record": record}

    # 2. Проверяем наличие конфликта по смыслу
    conflicting = [r for r in active if r.attribute == attribute and r.value != value]
    if not conflicting:
        record = memory.put(MemoryRecord(
            tenant_id=tenant_id, client_id=client_id, kind=kind, attribute=attribute,
            value=value, source=source, confidence=confidence,
            ticket_id=ticket_id, regulated=regulated), reason="новый факт")
        if verbose: print(f"  ДОБАВЛЕНО: {record.label()}")
        return {"action": "ADD", "record": record}

    existing = max(conflicting, key=lambda r: (r.trust, r.confidence))
    new_trust = SOURCE_TRUST[source]

    # 3. LLM оценивает смысл
    verdict = llm_reconcile(existing, attribute, value)
    if verbose:
        print(f"  Обнаружен конфликт: {existing.label()}")
        print(f"    новое значение: {value!r} ({SOURCE_TITLE[source]})")
        print(f"    Анализ модели: {verdict.proposed_action} — {verdict.reason}")

    # 4. Проверка правил доверия (Guardrail)
    if new_trust < existing.trust:
        claimed = memory.put(MemoryRecord(
            tenant_id=tenant_id, client_id=client_id, kind=kind,
            attribute=f"{attribute}__claimed", value=value, source=source,
            confidence=confidence, ticket_id=ticket_id), reason="зафиксировано расхождение")
        if verbose:
            print(f"    ПРАВИЛО: Доверие {new_trust} ниже текущего {existing.trust}. Замена запрещена.")
            print(f"    -> Сохранено как заявленное клиентом: {attribute}__claimed")
        return {"action": "CLAIM_RECORDED", "record": claimed}

    # Если доверие выше или равно, и модель подтверждает конфликт — обновляем
    existing.valid_to = now_iso()
    memory.update(existing, "INVALIDATE", f"заменено более надежным источником: {SOURCE_TITLE[source]}")
    record = memory.put(MemoryRecord(
        tenant_id=tenant_id, client_id=client_id, kind=kind, attribute=attribute,
        value=value, source=source, confidence=confidence,
        ticket_id=ticket_id, regulated=regulated), reason="обновление данных")
    if verbose: print(f"    -> СТАРОЕ АННУЛИРОВАНО + ДОБАВЛЕНО НОВОЕ: {record.label()}")
    return {"action": "INVALIDATE+ADD", "record": record}

**Пояснения к результату:**
- **Сценарий 1** - суть иерархии доверия. LLM могла предложить `UPDATE` (смысл-то противоречивый), но код отклонил: доверие 1 < 4. При этом расхождение **не выброшено** - оно сохранено как `card_product__claimed`. Это ровно то поведение, которое нужно: агент не поверил клиенту, не назвал его лжецом, и зафиксировал расхождение для оператора.
- **Сценарий 2** - доверие 3 против 2 (`client_statement` у существующей записи не выше `operator`). Здесь замена уже правомерна.
- **Сценарий 3** - АБС подтверждает. Реестр стал источником истины, цепочка сошлась.
- Старая запись `card_product=Классик` **не удалена**, а получила `valid_to`. Вопрос «какой продукт был у клиента в июле» по-прежнему имеет ответ - а именно он и понадобится при разборе июньской операции с занятия 7.
- Обратите внимание на разделение ответственности: `verdict.reason` - про смысл, `reason` в аудите - про политику. LLM никогда не решает вопросы регуляторного характера.

## Раздел 5. Удаление данных при обязанности хранить: tombstoning и маскирование

Клиент требует: "удалите все мои данные!". Наивная реализация делает DELETE FROM... – и создаёт нарушение закона: банк обязан хранить сведения об операциях (115-ФЗ, требования ЦБ, налоговое законодательство). Hard delete здесь неверен.

Правильная реализация разделяет данные на три категории:

| Категория | Что делаем | Почему |
|---|---|---|
| память агента (профиль, предпочтения, чат) | **удаляем** физически | нет основания хранить |
| сессионные данные | **удаляем** (или ждём TTL) | нет основания хранить |
| регулируемые сведения в БД (`regulated=True`) | **tombstone + маскирование** | обязаны хранить, но не должны использовать в процессах |

Для этого используется специальная метка-указатель **Tombstoning** - запись помечается удалённой, она исключается из всех чтений агента, но физически остаётся внутри нашей БД. Помимо этого применяется и **маскирование** - из записи вычищается все, что не требуется регулятору, остаётся минимальный набор данных.

При этом важно, чтобы подобные операции были выполнены для **всех** хранилищ: `Store`, векторный индекс (Qdrant), сессии (Redis).

Теперь давайте реализуем это в коде: пройдемся по всем хранилищам и удалим или скроем запись по требованию клиента.

In [15]:
from qdrant_client import QdrantClient, models
import fakeredis

# --- Векторный слой памяти: семантический поиск + точечное удаление по client_id ---
MEM_COLLECTION = "sber_memory"
qdrant = QdrantClient(location=":memory:")
qdrant.create_collection(
    collection_name=MEM_COLLECTION,
    vectors_config=models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE),
)
for field_name in ["tenant_id", "client_id", "attribute", "kind"]:
    qdrant.create_payload_index(collection_name=MEM_COLLECTION, field_name=field_name,
                                field_schema=models.PayloadSchemaType.KEYWORD)

_point_ids: dict[str, int] = {}
def _point_id(record_id: str) -> int:
    if record_id not in _point_ids:
        _point_ids[record_id] = len(_point_ids) + 1
    return _point_ids[record_id]

def vector_index(record: MemoryRecord):
    text = record_text(record.attribute, record.value)
    qdrant.upsert(collection_name=MEM_COLLECTION, points=[models.PointStruct(
        id=_point_id(record.record_id), vector=embed_cache.get(text),
        payload={"record_id": record.record_id, "tenant_id": record.tenant_id,
                 "client_id": record.client_id, "attribute": record.attribute,
                 "value": record.value, "kind": record.kind, "source": record.source,
                 "created_at": record.created_at, "text": text})])

for record in memory.active(TENANT, CLIENT) + memory.active(TENANT, OTHER_CLIENT):
    vector_index(record)
print(f"В векторный слой памяти проиндексировано точек: {qdrant.count(MEM_COLLECTION).count}")

# --- Сессионный слой с TTL ---
redis_client = fakeredis.FakeStrictRedis(decode_responses=True)
SESSION_TTL = 3600 * 6
def session_key(tenant_id: str, client_id: str) -> str:
    return f"{tenant_id}:session:{client_id}"

redis_client.setex(session_key(TENANT, CLIENT), SESSION_TTL, json.dumps(
    {"draft": {"operation_date": "2026-06-14", "amount": 12500.0, "channel": "банкомат"},
     "status": "собираем реквизиты"}, ensure_ascii=False))
print(f"Сессия в Redis создана, TTL {redis_client.ttl(session_key(TENANT, CLIENT))} с")

# --- Маскирование и каскадное удаление -----------------------------------------
REGULATED_KEEP_FIELDS = {"record_id", "tenant_id", "client_id", "kind", "ticket_id",
                         "created_at", "valid_from", "valid_to", "source", "regulated",
                         "schema_version", "attribute"}

def mask_regulated(record: MemoryRecord) -> MemoryRecord:
    """Оставляем минимум, требуемый регулятором: факт обращения и его реквизиты, без содержания."""
    record.value = f"<маскировано по запросу клиента {NOW.date().isoformat()}>"
    record.confidence = 0.0
    record.tombstoned = True
    return record

def erase_client_memory(tenant_id: str, client_id: str,
                        actor: str = "dpo-officer") -> dict:
    """Удаление данных клиента с учётом обязанности хранить регулируемые сведения.
    Каскад: Store -> векторный индекс -> сессии."""
    report = {"purged": [], "masked": [], "stores_touched": []}

    # 1. Слой Store
    for record in memory.all_records(tenant_id, client_id):
        if record.regulated:
            masked = mask_regulated(record)
            memory.update(masked, "MASK_REGULATED",
                          "запрос клиента на удаление; сведения об операции сохранены "
                          "в силу требований законодательства", )
            AUDIT[-1]["actor"] = actor
            report["masked"].append(f"{record.kind}/{record.attribute} (тикет {record.ticket_id})")
        else:
            record.tombstoned = True
            memory.update(record, "PURGE", "запрос клиента на удаление; основания для хранения нет")
            AUDIT[-1]["actor"] = actor
            report["purged"].append(f"{record.kind}/{record.attribute}")
    report["stores_touched"].append("LangGraph Store")

    # 2. Векторный индекс - точечное удаление по фильтру client_id
    before = qdrant.count(MEM_COLLECTION).count
    qdrant.delete(collection_name=MEM_COLLECTION, points_selector=models.FilterSelector(
        filter=models.Filter(must=[
            models.FieldCondition(key="tenant_id", match=models.MatchValue(value=tenant_id)),
            models.FieldCondition(key="client_id", match=models.MatchValue(value=client_id)),
        ])))
    after = qdrant.count(MEM_COLLECTION).count
    report["stores_touched"].append(f"Qdrant (удалено векторов: {before - after})")

    # 3. Сессии
    removed = redis_client.delete(session_key(tenant_id, client_id))
    report["stores_touched"].append(f"Redis (удалено ключей сессии: {removed})")
    return report

print("\n" + "=" * 100)
print(f"ЗАПРОС КЛИЕНТА {CLIENT} НА УДАЛЕНИЕ ДАННЫХ")
print("=" * 100)
report = erase_client_memory(TENANT, CLIENT)

print("\nВЫЧИЩЕНО ПОЛНОСТЬЮ (нет основания хранить):")
for item in report["purged"]:
    print(f"  - {item}")
print("\nСОХРАНЕНО И МАСКИРОВАНО (обязанность хранить по требованиям регулятора):")
for item in report["masked"]:
    print(f"  - {item}")
print("\nЗАТРОНУТЫЕ ХРАНИЛИЩА:")
for item in report["stores_touched"]:
    print(f"  - {item}")

print(f"\nАктивная память клиента после удаления: {len(memory.active(TENANT, CLIENT))} записей")
print(f"Физически осталось записей в Store: {len(memory.all_records(TENANT, CLIENT))} "
      f"(все помечены tombstoned)")
print(f"\nПРОВЕРКА, что другой клиент не затронут: "
      f"{len(memory.active(TENANT, OTHER_CLIENT))} активных записей у {OTHER_CLIENT}, "
      f"векторов в индексе {qdrant.count(MEM_COLLECTION).count}")

print("\nЧТО ОСТАЛОСЬ ИЗ РЕГУЛИРУЕМОГО (содержание вычищено, факт обращения сохранён):")
for record in memory.all_records(TENANT, CLIENT):
    if record.regulated:
        print(f"  {record.kind}/{record.attribute}: value={record.value!r}, "
              f"ticket_id={record.ticket_id}, created_at={record.created_at[:10]}")

/tmp/ipykernel_2663801/2129893896.py:12: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  qdrant.create_payload_index(collection_name=MEM_COLLECTION, field_name=field_name,


В векторный слой памяти проиндексировано точек: 7
Сессия в Redis создана, TTL 21600 с

ЗАПРОС КЛИЕНТА client-4471 НА УДАЛЕНИЕ ДАННЫХ

ВЫЧИЩЕНО ПОЛНОСТЬЮ (нет основания хранить):
  - profile/card_product
  - profile/deposit_product
  - profile/region
  - profile/preferred_channel
  - profile/app_setup

СОХРАНЕНО И МАСКИРОВАНО (обязанность хранить по требованиям регулятора):
  - episode/dispute_2026_06 (тикет T-2026-06-3312)
  - episode/dispute_2025_09 (тикет T-2025-09-0071)

ЗАТРОНУТЫЕ ХРАНИЛИЩА:
  - LangGraph Store
  - Qdrant (удалено векторов: 6)
  - Redis (удалено ключей сессии: 1)

Активная память клиента после удаления: 0 записей
Физически осталось записей в Store: 7 (все помечены tombstoned)

ПРОВЕРКА, что другой клиент не затронут: 1 активных записей у client-9902, векторов в индексе 1

ЧТО ОСТАЛОСЬ ИЗ РЕГУЛИРУЕМОГО (содержание вычищено, факт обращения сохранён):
  episode/dispute_2026_06: value='<маскировано по запросу клиента 2026-08-13>', ticket_id=T-2026-06-3312, created_at=2

/tmp/ipykernel_2663801/2129893896.py:40: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  redis_client.setex(session_key(TENANT, CLIENT), SESSION_TTL, json.dumps(


**Пояснения к результату:**
- Обратите внимание, **что именно** осталось от регулируемого эпизода: тикет, дата, источник, факт обращения. Содержание - маскировано. Регулятор может подтвердить, что обращение было и когда; агент этими данными пользоваться не может.
- Каскад прошёл по трём хранилищам, и по каждому отчёт содержит число затронутых объектов. Отчёт - это не логирование для красоты: при проверке нужно предъявить доказательство, что удаление произошло во **всех** системах.
- Проверка «другой клиент не затронут» обязательна. Фильтр `FilterSelector` по `client_id` легко написать слишком широко, и такая ошибка уносит данные соседних клиентов молча.
- Различайте `PURGE` и `MASK_REGULATED` в аудите. При проверке вопрос будет звучать не «удалили ли вы», а «на каком основании вы сохранили вот это» - и ответ должен быть в логе.

## Раздел 6. Семантический поиск по памяти с recency-взвешиванием

При чтении памяти похожесть - необходимое, но недостаточное условие. Факт может идеально соответствовать запросу и быть **устаревшим**. Поэтому финальный ранг комбинирует два сигнала:

$$\text{score} = w_{sim} \cdot \text{similarity} + w_{rec} \cdot 2^{-\text{age}/\text{half life}}$$

Экспоненциальное затухание с периодом полураспада - стандартный приём. `half_life` — это **продуктовое решение**:
- для ассистента поддержки короткий период оправдан: обстоятельства клиента меняются;
- для персонального помощника агрессивное затухание вредно: «предпочитаю окно у прохода» не устаревает годами.

Retrieval-конвейер Zep устроен так же: cosine + BM25 + обход графа, слияние через RRF/MMR/cross-encoder. Qdrant позволяет вынести recency-бустинг на сторону хранилища через Formula Query. Мы считаем на клиенте - прозрачнее для понимания.

Восстановим память клиента (после удаления она пуста) и посмотрим на поиск.

In [16]:
# Восстанавливаем память заново — предыдущий раздел её вычистил.
memory = MemoryStore()
AUDIT.clear()
for record in seed:
    fresh = record.model_copy(deep=True)
    fresh.tombstoned = False
    fresh.valid_to = None
    memory.put(fresh, reason="повторная загрузка для демонстрации поиска")
memory.put(MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="profile",
                        attribute="card_product", value="Премиум", source="abs",
                        confidence=0.99, created_at="2026-08-10T10:00:00+00:00"),
           reason="актуальный продукт из АБС")
memory.put(MemoryRecord(tenant_id=TENANT, client_id=CLIENT, kind="profile",
                        attribute="accessibility", value="плохо слышит, телефон затруднён",
                        source="operator", confidence=0.9,
                        created_at="2026-03-01T10:00:00+00:00"),
           reason="отмечено оператором")

SIM_FLOOR = 0.55        # порог: ниже — вообще не рассматриваем
W_SIM, W_REC = 0.75, 0.25
HALF_LIFE_DAYS = {"profile": 720, "episode": 120, "transient": 3}

def recency(record: MemoryRecord, at: Optional[datetime] = None) -> float:
    at = at or NOW
    age_days = max((at - datetime.fromisoformat(record.created_at)).days, 0)
    return 0.5 ** (age_days / HALF_LIFE_DAYS[record.kind])

def search_memory(tenant_id: str, client_id: str, query: str, top_k: int = 4,
                  sim_floor: float = SIM_FLOOR, verbose: bool = True) -> list[dict]:
    """Семантический поиск по памяти: порог сходства + взвешивание по свежести."""
    t0 = time.perf_counter()
    qvec = embed_cache.get(query)
    scored = []
    for record in memory.active(tenant_id, client_id):
        sim = cosine(qvec, embed_cache.get(record_text(record.attribute, record.value)))
        if sim < sim_floor:
            continue
        rec = recency(record)
        scored.append({"record": record, "similarity": sim, "recency": rec,
                       "score": W_SIM * sim + W_REC * rec})
    scored.sort(key=lambda x: -x["score"])
    latency_ms = (time.perf_counter() - t0) * 1000
    if verbose:
        print(f"ЗАПРОС: {query}")
        print(f"  рассмотрено активных записей: {len(memory.active(tenant_id, client_id))}, "
              f"прошло порог {sim_floor}: {len(scored)}, латентность {latency_ms:.1f} мс")
        for item in scored[:top_k]:
            r = item["record"]
            age = (NOW - datetime.fromisoformat(r.created_at)).days
            print(f"    score={item['score']:.3f} (sim={item['similarity']:.3f} "
                  f"rec={item['recency']:.3f}, {age:>4} дн.) [{r.kind:<7}] "
                  f"{r.attribute}={r.value[:44]!r}")
    return scored[:top_k]

for query in ["как лучше связаться с этим клиентом",
              "какой у клиента продукт",
              "были ли раньше споры по операциям"]:
    search_memory(TENANT, CLIENT, query)
    print()

print("=" * 100)
print("ВЛИЯНИЕ HALF_LIFE: тот же запрос при агрессивном затухании профиля")
print("=" * 100)
saved = HALF_LIFE_DAYS["profile"]
for half_life in [720, 90, 30]:
    HALF_LIFE_DAYS["profile"] = half_life
    top = search_memory(TENANT, CLIENT, "как лучше связаться с этим клиентом",
                        top_k=2, verbose=False)
    head = ", ".join(f"{i['record'].attribute}({i['score']:.3f})" for i in top)
    print(f"  half_life={half_life:>3} дн. -> {head}")
HALF_LIFE_DAYS["profile"] = saved
print("\nЧем короче период полураспада, тем сильнее свежесть перебивает смысловую близость.")
print("Для поддержки это разумно; для персонального помощника — способ забыть то, что не устаревает.")

ЗАПРОС: как лучше связаться с этим клиентом
  рассмотрено активных записей: 8, прошло порог 0.55: 8, латентность 708.4 мс
    score=0.852 (sim=0.803 rec=1.000,    0 дн.) [profile] preferred_channel='только чат, звонки не принимает'
    score=0.839 (sim=0.785 rec=1.000,    0 дн.) [profile] region='Новосибирская область'
    score=0.837 (sim=0.783 rec=1.000,    0 дн.) [profile] card_product='Классик'
    score=0.831 (sim=0.775 rec=0.997,    3 дн.) [profile] card_product='Премиум'

ЗАПРОС: какой у клиента продукт
  рассмотрено активных записей: 8, прошло порог 0.55: 8, латентность 197.3 мс
    score=0.872 (sim=0.829 rec=1.000,    0 дн.) [profile] card_product='Классик'
    score=0.868 (sim=0.825 rec=0.997,    3 дн.) [profile] card_product='Премиум'
    score=0.843 (sim=0.790 rec=1.000,    0 дн.) [profile] deposit_product='Сбер-Доход, открыт 2025-11-04'
    score=0.830 (sim=0.773 rec=1.000,    0 дн.) [profile] app_setup='мобильное приложение 8.2.1, Android 14'

ЗАПРОС: были ли раньше споры

**Пояснения к результату:**
- Запрос «как лучше связаться» поднимает и `preferred_channel`, и `accessibility` — семантически близкие, но разные факты. Хорошо, что мы их **не склеили** дедупликацией в разделе 3: оператору нужны оба.
- `card_product=Премиум` (свежая запись из АБС) обходит инвалидированную `Классик` — та вообще не попала в выдачу, потому что `is_active()` её отфильтровал. Инвалидация и поиск работают согласованно.
- Порог `sim_floor` отсекает шум **до** взвешивания. Без него в выдачу с ненулевым весом попадёт вообще всё, и recency начнёт вытягивать свежий, но нерелевантный факт наверх.
- Латентность поиска печатается специально: чтение памяти происходит на **каждом** ходу диалога и входит в бюджет ответа. Это тема следующего раздела.

## Раздел 7. Фоновая консолидация и бюджет чтения памяти

Два последних инженерных вопроса.

**Первый: где выполнять дорогую переработку памяти.** Разделяем два пути записи:
- **hot path** — то, что должно произойти немедленно, в ходе диалога: дедупликация, и разрешение противоречий для критичных фактов;
- **background** — то, что можно отложить: слияние близких фактов, порождение обобщений, пересчёт профиля.

Это идея **sleep-time compute** (Letta, апрель 2025): агент «во сне» перерабатывает накопленное, чтобы в горячем пути отвечать быстрее. Реализуем отдельным графом LangGraph, который вызывается **вне** диалога.

**Второй: сколько стоит чтение памяти.** Каждый ход диалога включает эмбеддинг запроса, обход памяти, отбор и вставку в контекст. Это латентность и токены, и у них должен быть **бюджет** — как и у контекста на занятии 6.

In [17]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class ConsolidationState(TypedDict):
    tenant_id: str
    client_id: str
    merged: list
    summary: str
    stats: dict

CONSOLIDATE_PROMPT = """Ты консолидируешь память ассистента поддержки банка «Сбер».
Ниже — факты о клиенте. Составь краткую справку для оператора: 3-4 пункта, только на основе
приведённых фактов, без домыслов. Отдельно отметь расхождения, если атрибут с суффиксом
__claimed противоречит авторитетной записи.

ФАКТЫ:
{facts}"""

def node_merge_duplicates(state: ConsolidationState) -> dict:
    """Фоновое слияние семантических дублей — то, что незачем делать в горячем пути."""
    records = memory.active(state["tenant_id"], state["client_id"])
    merged, seen = [], []
    for record in sorted(records, key=lambda r: (-r.trust, -r.confidence)):
        dup = find_duplicate(record.attribute, record.value, seen)
        if dup and dup[0].attribute == record.attribute:
            keeper, sim = dup
            record.valid_to = now_iso()
            memory.update(record, "MERGE_DUPLICATE",
                          f"слито в record_id={keeper.record_id} при фоновой консолидации, "
                          f"сходство {sim:.3f}")
            merged.append(f"{record.attribute}={record.value[:40]!r} -> {keeper.record_id}")
        else:
            seen.append(record)
    return {"merged": merged}

def node_build_summary(state: ConsolidationState) -> dict:
    """Порождение обобщающей справки. Дорого, поэтому — вне горячего пути."""
    records = memory.active(state["tenant_id"], state["client_id"])
    facts = "\n".join(f"- {r.attribute} = {r.value} (источник: {SOURCE_TITLE[r.source]})"
                      for r in records)
    try:
        summary = llm.invoke([HumanMessage(CONSOLIDATE_PROMPT.format(facts=facts))]).content.strip()
    except Exception as exc:
        summary = f"(консолидация не выполнена: {type(exc).__name__})"
    consolidated = MemoryRecord(
        tenant_id=state["tenant_id"], client_id=state["client_id"], kind="profile",
        attribute="operator_briefing", value=summary, source="operator", confidence=0.8)
    memory.put(consolidated, reason="фоновая консолидация (sleep-time)")
    return {"summary": summary}

def node_stats(state: ConsolidationState) -> dict:
    records = memory.all_records(state["tenant_id"], state["client_id"])
    return {"stats": {
        "всего записей": len(records),
        "активных": sum(1 for r in records if r.is_active()),
        "инвалидировано": sum(1 for r in records if r.valid_to),
        "регулируемых": sum(1 for r in records if r.regulated),
        "слито дублей": len(state.get("merged", [])),
    }}

consolidation = StateGraph(ConsolidationState)
consolidation.add_node("merge_duplicates", node_merge_duplicates)
consolidation.add_node("build_summary", node_build_summary)
consolidation.add_node("stats", node_stats)
consolidation.add_edge(START, "merge_duplicates")
consolidation.add_edge("merge_duplicates", "build_summary")
consolidation.add_edge("build_summary", "stats")
consolidation.add_edge("stats", END)
consolidation_app = consolidation.compile()

print("Фоновая консолидация (вызывается ВНЕ диалога, в простое):\n")
t0 = time.time()
out = consolidation_app.invoke({"tenant_id": TENANT, "client_id": CLIENT,
                                "merged": [], "summary": "", "stats": {}})
print(f"Выполнено за {time.time() - t0:.1f} с\n")
if out["merged"]:
    print("Слитые дубли:")
    for item in out["merged"]:
        print(f"  - {item}")
else:
    print("Дублей для слияния не найдено.")
print("\nСправка для оператора (результат консолидации):")
print(textwrap.fill(out["summary"], 100, initial_indent="  ", subsequent_indent="  "))
print("\nСтатистика памяти:")
for key, value in out["stats"].items():
    print(f"  {key}: {value}")

Фоновая консолидация (вызывается ВНЕ диалога, в простое):

Выполнено за 3.8 с

Слитые дубли:
  - card_product='Премиум' -> d5eed551c00a

Справка для оператора (результат консолидации):
  ## Справка по клиенту  **Основные сведения:**  - **Тип карты:** Классик   - **Депозит:** Сбер-
  Доход, дата открытия: 04.11.2025   - **Регион проживания:** Новосибирская область   -
  **Предпочитаемый канал связи:** исключительно чат, звонки не принимает    **Расхождения:** Нет
  зафиксированных расхождений между заявленными клиентом фактами и информацией из реестра.
  **Дополнительная информация:** Клиент ранее обращался с претензией относительно комиссии в размере
  250 рублей за операцию снятия наличных, произошедшую 14 июня 2026 года. Клиент имеет ограничения
  слуха, использование телефона затруднено. Используемое мобильное приложение версии 8.2.1 на
  платформе Android 14.

Статистика памяти:
  всего записей: 10
  активных: 8
  инвалидировано: 1
  регулируемых: 2
  слито дублей: 1


In [18]:
# --- Бюджет latency и стоимости чтения памяти ---
READ_BUDGET_MS = 150      # сколько мы готовы потратить на чтение памяти в одном ходу
READ_BUDGET_TOKENS = 300  # сколько токенов памяти готовы отдать контексту

def memory_block_for_context(tenant_id: str, client_id: str, query: str,
                             budget_tokens: int = READ_BUDGET_TOKENS) -> tuple[str, dict]:
    """Готовит блок памяти для вставки в контекст, укладываясь в бюджет токенов."""
    t0 = time.perf_counter()
    hits = search_memory(tenant_id, client_id, query, top_k=8, verbose=False)
    lines, used = [], 0
    for item in hits:
        record = item["record"]
        line = f"- {record.attribute}: {record.value} ({SOURCE_TITLE[record.source]})"
        cost = max(len(line) // 3, 1)     # грубая оценка; точный подсчёт — llm.get_num_tokens
        if used + cost > budget_tokens:
            break
        lines.append(line)
        used += cost
    latency_ms = (time.perf_counter() - t0) * 1000
    block = ("ПАМЯТЬ О КЛИЕНТЕ:\n" + "\n".join(lines)) if lines else ""
    return block, {"latency_ms": latency_ms, "tokens_used": used,
                   "candidates": len(hits), "included": len(lines),
                   "cache_calls": embed_cache.calls}

print(f"{'запрос':<44} {'(мс)':>7} {'токенов':>5} {'кандидатов':>5} {'вкл':>4}  в бюджете?")
print("-" * 88)
for query in ["как связаться с клиентом", "какой продукт у клиента",
              "история споров по операциям", "настройки приложения клиента"]:
    block, stats = memory_block_for_context(TENANT, CLIENT, query)
    ok = "да" if stats["latency_ms"] <= READ_BUDGET_MS and stats["tokens_used"] <= READ_BUDGET_TOKENS else "НЕТ"
    print(f"{query[:43]:<44} {stats['latency_ms']:>7.1f} {stats['tokens_used']:>5} "
          f"{stats['candidates']:>5} {stats['included']:>4}  {ok}")

print(f"\nВызовов эмбеддера за всё занятие: {embed_cache.calls} "
      f"(кэш содержит {len(embed_cache.cache)} векторов)")
print("Без кэша каждое чтение памяти на каждой реплике диалога — это отдельный сетевой вызов.")
print(f"\nБюджеты: латентность <= {READ_BUDGET_MS} мс, токены <= {READ_BUDGET_TOKENS}.")
print("Чтение памяти отрезается по бюджету так же, как история диалога на занятии 6:")
print("сначала самое релевантное, остальное не попадает в контекст вовсе.")

print("\n\nПример готового блока памяти для вставки в промпт:")
block, _ = memory_block_for_context(TENANT, CLIENT, "как связаться с клиентом")
print(block)

запрос                                          (мс) токенов кандидатов  вкл  в бюджете?
----------------------------------------------------------------------------------------
как связаться с клиентом                       399.8   286     8    4  НЕТ
какой продукт у клиента                        305.7   290     8    4  НЕТ
история споров по операциям                    207.3   286     8    4  НЕТ
настройки приложения клиента                   204.6   293     8    4  НЕТ

Вызовов эмбеддера за всё занятие: 20 (кэш содержит 20 векторов)
Без кэша каждое чтение памяти на каждой реплике диалога — это отдельный сетевой вызов.

Бюджеты: латентность <= 150 мс, токены <= 300.
Чтение памяти отрезается по бюджету так же, как история диалога на занятии 6:
сначала самое релевантное, остальное не попадает в контекст вовсе.


Пример готового блока памяти для вставки в промпт:
ПАМЯТЬ О КЛИЕНТЕ:
- preferred_channel: только чат, звонки не принимает (заявление клиента)
- operator_briefing: ## Справка п

**Пояснения к результату:**
- Консолидация вынесена в **отдельный граф**, который не вызывается из диалога. Это и есть смысл sleep-time compute: дорогое обобщение выполняется в простое, а горячий путь читает готовый результат.
- Справка `operator_briefing` — производная запись. Она удобна, но у неё есть цена: если исходные факты изменятся, справка устареет молча. В проде такие записи нужно инвалидировать при изменении источников — иначе получите свежий ответ на основе старого обобщения.
- Бюджет чтения памяти работает точно так же, как бюджет контекста на занятии 6: **сначала самое релевантное, остальное не попадает вовсе**. Кандидатов может быть восемь, а в контекст войдут три.
- Счётчик кэша показывает реальную экономию. Чтение памяти на каждом ходу без кэша эмбеддингов — самая частая причина, по которой «агент с памятью» отвечает вдвое медленнее агента без неё.

## Практика

Собираем сквозной сценарий целиком: клиент обращается заново, память читается, приходит противоречивый факт, срабатывает модуль согласования фактов, затем клиент требует удалить данные — и мы предъявляем полный audit trail с разделением «вычищено / сохранено по требованиям регулятора».

Это **итоговый артефакт всего блока памяти** курса.

In [19]:
# --- Сброс состояния и чистый прогон сквозного сценария ---
memory = MemoryStore()
AUDIT.clear()
qdrant.delete_collection(MEM_COLLECTION)
qdrant.create_collection(collection_name=MEM_COLLECTION,
    vectors_config=models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE))
for field_name in ["tenant_id", "client_id", "attribute", "kind"]:
    qdrant.create_payload_index(collection_name=MEM_COLLECTION, field_name=field_name,
                                field_schema=models.PayloadSchemaType.KEYWORD)

print("ШАГ 1. Загрузка профиля из АБС и предыдущих обращений")
print("=" * 100)
for record in seed:
    fresh = record.model_copy(deep=True)
    fresh.tombstoned, fresh.valid_to = False, None
    memory.put(fresh, reason="загрузка при открытии обращения")
    vector_index(fresh)
redis_client.setex(session_key(TENANT, CLIENT), SESSION_TTL,
                   json.dumps({"status": "новое обращение"}, ensure_ascii=False))
for record in memory.active(TENANT, CLIENT):
    print(f"  [{record.kind:<8}] {record.label()}")

print("\n\nШАГ 2. Чтение памяти в начале обращения (точка чтения из профиля памяти)")
print("=" * 100)
block, stats = memory_block_for_context(TENANT, CLIENT, "клиент пишет по поводу комиссии за снятие")
print(block)
print(f"\n  латентность {stats['latency_ms']:.1f} мс, токенов {stats['tokens_used']}, "
      f"кандидатов {stats['candidates']}, включено {stats['included']}")

print("\n\nШАГ 3. Клиент утверждает: «у меня же Премиум, комиссии быть не должно»")
print("=" * 100)
result = ingest_fact(TENANT, CLIENT, "profile", "card_product", "Премиум",
                     source="chat", confidence=0.7)

print("\n\nШАГ 4. Что агент должен ответить клиенту")
print("=" * 100)
authoritative = [r for r in memory.active(TENANT, CLIENT) if r.attribute == "card_product"]
claimed = [r for r in memory.active(TENANT, CLIENT) if r.attribute == "card_product__claimed"]
context = (f"Реестр продуктов (АБС): {authoritative[0].value if authoritative else 'нет данных'}. "
           f"Клиент утверждает: {claimed[0].value if claimed else 'нет расхождений'}.")
answer = llm.invoke([
    SystemMessage(
        "Ты — ассистент поддержки банка Сбер. Клиент утверждает одно, реестр показывает другое. "
        "Не спорь с клиентом и не называй его неправым, но и не подтверждай неверные сведения. "
        "Объясни расхождение и предложи проверку у оператора. Два-три предложения, по-русски."),
    HumanMessage(context)]).content
print(f"  Контекст для модели: {context}")
print(f"\n  ОТВЕТ АГЕНТА:\n{textwrap.fill(answer, 96, initial_indent='  ', subsequent_indent='  ')}")

print("\n\nШАГ 5. Клиент требует удалить свои данные")
print("=" * 100)
report = erase_client_memory(TENANT, CLIENT, actor="dpo-officer")
print("  ВЫЧИЩЕНО ПОЛНОСТЬЮ:")
for item in report["purged"]:
    print(f"    - {item}")
print("  СОХРАНЕНО ПО ТРЕБОВАНИЯМ РЕГУЛЯТОРА (маскировано):")
for item in report["masked"]:
    print(f"    - {item}")
print("  ХРАНИЛИЩА В КАСКАДЕ:")
for item in report["stores_touched"]:
    print(f"    - {item}")

ШАГ 1. Загрузка профиля из АБС и предыдущих обращений


/tmp/ipykernel_2663801/1600318845.py:18: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  redis_client.setex(session_key(TENANT, CLIENT), SESSION_TTL,


  [profile ] card_product='Классик' (АБС (реестр), доверие 4, conf 0.99)
  [profile ] deposit_product='Сбер-Доход, открыт 2025-11-04' (АБС (реестр), доверие 4, conf 0.99)
  [profile ] region='Новосибирская область' (АБС (реестр), доверие 4, conf 0.99)
  [profile ] preferred_channel='только чат, звонки не принимает' (заявление клиента, доверие 2, conf 0.90)
  [profile ] app_setup='мобильное приложение 8.2.1, Android 14' (слова в чате, доверие 1, conf 0.60)
  [episode ] dispute_2026_06='оспаривал комиссию 250 ₽ за снятие 14.06.2026' (оператор, доверие 3, conf 0.95) [REG]


ШАГ 2. Чтение памяти в начале обращения (точка чтения из профиля памяти)
ПАМЯТЬ О КЛИЕНТЕ:
- card_product: Классик (АБС (реестр))
- preferred_channel: только чат, звонки не принимает (заявление клиента)
- region: Новосибирская область (АБС (реестр))
- deposit_product: Сбер-Доход, открыт 2025-11-04 (АБС (реестр))
- dispute_2026_06: оспаривал комиссию 250 ₽ за снятие 14.06.2026 (оператор)
- app_setup: мобильное приложени

In [20]:
# --- ИТОГОВЫЙ АРТЕФАКТ: полный audit trail ---
print("=" * 108)
print("AUDIT TRAIL ПО КЛИЕНТУ", CLIENT)
print("=" * 108)
print(f"{'#':>3} {'действие':<17} {'актор':<12} {'вид':<9} {'атрибут':<26} {'обоснование'}")
print("-" * 108)
for i, entry in enumerate([e for e in AUDIT if e["client_id"] == CLIENT], 1):
    print(f"{i:>3} {entry['action']:<17} {entry['actor']:<12} {entry['kind']:<9} "
          f"{entry['attribute'][:25]:<26} {entry['reason'][:44]}")

print("\n" + "=" * 108)
print("СВОДКА ПО ДЕЙСТВИЯМ")
print("=" * 108)
for action, count in Counter(e["action"] for e in AUDIT if e["client_id"] == CLIENT).most_common():
    print(f"  {action:<18} {count}")

print("\n" + "=" * 108)
print("РАЗДЕЛЕНИЕ «ВЫЧИЩЕНО / СОХРАНЕНО» — то, что предъявляется при проверке")
print("=" * 108)
records = memory.all_records(TENANT, CLIENT)
print(f"{'вид':<9} {'атрибут':<28} {'сост.':<12} {'рег.':<5} {'тикет':<18} {'значение'}")
print("-" * 108)
for record in sorted(records, key=lambda r: (not r.regulated, r.kind, r.attribute)):
    state = "МАСКИРОВАНО" if (record.regulated and record.tombstoned) else (
            "удалено" if record.tombstoned else "активна")
    print(f"{record.kind:<9} {record.attribute[:27]:<28} {state:<12} "
          f"{'да' if record.regulated else '—':<5} {(record.ticket_id or '—'):<18} "
          f"{record.value[:32]}")

print(f"\nАктивных записей в памяти агента: {len(memory.active(TENANT, CLIENT))} "
      f"(агент больше ничего не знает об этом клиенте)")
print(f"Сохранено регулируемых сведений: "
      f"{sum(1 for r in records if r.regulated)} (факт обращения без содержания)")
print(f"Записей в аудите: {len([e for e in AUDIT if e['client_id'] == CLIENT])} "
      f"(аудит не удаляется — он и есть доказательство корректности удаления)")

AUDIT TRAIL ПО КЛИЕНТУ client-4471
  # действие          актор        вид       атрибут                    обоснование
------------------------------------------------------------------------------------------------------------
  1 PUT               agent        profile   card_product               загрузка при открытии обращения
  2 PUT               agent        profile   deposit_product            загрузка при открытии обращения
  3 PUT               agent        profile   region                     загрузка при открытии обращения
  4 PUT               agent        profile   preferred_channel          загрузка при открытии обращения
  5 PUT               agent        episode   dispute_2026_06            загрузка при открытии обращения
  6 PUT               agent        episode   dispute_2025_09            загрузка при открытии обращения
  7 PUT               agent        profile   app_setup                  загрузка при открытии обращения
  8 PUT               agent        profile  

### Задания для самостоятельной работы

1. **Смените профиль сценария.** Спроектируйте `RETENTION_DAYS` и `HALF_LIFE_DAYS` под **персонального помощника** и прогоните разделы 6–7. Какие факты, полезные помощнику, выпадают при профиле поддержки — и почему это ошибка проектирования, а не настройки?
2. **Требуйте двух подтверждений.** Замените политику в `ingest_fact`: замена авторитетного факта возможна только при **двух независимых** источниках одного уровня доверия. Как изменится audit trail в сценарии 2?
3. **Сломайте guardrail.** Уберите проверку `new_trust < existing.trust` и отдайте решение целиком LLM. Прогоните сценарий 1 несколько раз. Стабилен ли результат? Почему регуляторное правило нельзя оставлять на усмотрение модели?
4. **Найдите порог дедупликации.** Постройте таблицу вердиктов для `DEDUP_THRESHOLD` от 0.70 до 0.95 с шагом 0.05 на четырёх кандидатах из раздела 3. При каком значении «плохо слышу» склеивается с «не принимаю звонки»?
5. **Забудьте хранилище.** Удалите из `erase_client_memory` шаг с Qdrant и напишите проверку, которая эту утечку **поймает**: после удаления семантический поиск по вектору не должен находить данные клиента.
6. **Инвалидируйте производное.** Сделайте так, чтобы `operator_briefing` автоматически получал `valid_to` при изменении любого факта, на котором он построен. Что для этого нужно добавить в схему записи?
7. **Би-темпоральный запрос.** Напишите функцию `as_of(tenant_id, client_id, at: datetime)`, возвращающую состояние памяти **на произвольный момент в прошлом** — то, что мы знали о клиенте в июне. Проверьте ее на записи `card_product`, у которой есть `valid_to`.
8. **Замерьте бюджет.** Прогоните сорокаходовой диалог с занятия 6, вызывая `memory_block_for_context` на каждом ходу. Какова суммарная латентность чтения памяти и сколько токенов она съела?

## Итоги

В этом уроке мы собрали архитектуру памяти для поддержки:

1. Определили профиль памяти: типы записей, сроки хранения и точки записи/чтения.
2. Спроектировали схему записи с двумя осями времени, источником, уверенностью и regulated.
3. Добавили изоляцию клиентов через namespaces и проверили её тестом.
4. Настроили дедупликацию и разобрали, почему здесь лучше ошибиться в сторону лишнего дубля, чем склеить разные факты.
5. Реализовали реконсиляцию конфликтов: LLM оценивает смысл, а политика приоритетов остаётся в коде.
6. Использовали инвалидацию вместо удаления (valid_to), чтобы сохранять историю состояния памяти.
7. Разобрали удаление и хранение по требованиям регулятора: tombstone, маскирование и каскад по всем хранилищам.
8. Настроили семантический поиск с учётом сходства и давности.
9. Вынесли консолидацию в фон и ограничили бюджет latency и токенов на чтение памяти.
10. В конце свели всё в audit trail — итоговый артефакт блока о памяти.

## Что запомнить

1. Профиль памяти проектируем до кода: что храним, как долго, откуда берём записи и когда читаем.
2. Источник и уверенность — часть данных. source и confidence нужны, чтобы разбираться с конфликтами, а не только для отладки.
3. LLM предлагает, код решает. Смысл фактов можно оценивать моделью, но приоритеты и регуляторные правила должны оставаться детерминированными.
4. Историю лучше сохранять, а не стирать. valid_to позволяет восстановить состояние памяти в прошлом. При этом регулируемые данные могут требовать маскирования вместо hard delete, а удаление должно каскадировать по всем хранилищам.
5. С памятью лучше быть консервативным. Высокий порог дедупликации защищает от потери разных фактов, а half_life стоит рассматривать как продуктовое решение, а не просто технический параметр.
6. Память имеет стоимость. Поиск тратит latency и токены, поэтому тяжелые операции лучше выносить в фон, а повторяющиеся вычисления — кэшировать.


## Полезные материалы

- [mem0ai/mem0](https://github.com/mem0ai/mem0) – Apache-2.0, статья ECAI 2025 (arXiv:2504.19413); это основной источник логики организации памяти.
- [mem0 `configs/prompts.py`](https://github.com/mem0ai/mem0/blob/main/mem0/configs/prompts.py) – `DEFAULT_UPDATE_MEMORY_PROMPT` с операциями ADD/UPDATE/DELETE/NONE, пример как организация памяти устроена внутри mem0
- [mem0 `memory/main.py`](https://github.com/mem0ai/mem0/blob/main/mem0/memory/main.py) – конвейер записи, а также реализация функций `delete` / `delete_all`.
- [Zep: A Temporal Knowledge Graph Architecture for Agent Memory](https://arxiv.org/abs/2501.13956) – arXiv:2501.13956; подход к организации памяти через Temporal Knowledge Graph, `t_valid`/`t_invalid`, retrieval-конвейер cosine + BM25 + обход графа.
- [getzep/graphiti](https://github.com/getzep/graphiti) – реализация темпорального графа знаний с использованием Neo4j / FalkorDB / Kuzu, может служить отличным примером обновления памяти в реальном времени.
- [Graphiti knowledge graph memory — разбор Neo4j](https://neo4j.com/blog/developer/graphiti-knowledge-graph-memory/) – доступное объяснение, зачем памяти агента необходим граф.
- [Letta sleep-time compute](https://www.letta.com/blog/sleep-time-compute/) – идея фоновой переработки памяти во время простоя.
- [letta-ai/sleep-time-compute](https://github.com/letta-ai/sleep-time-compute) – код к статье.
- [langchain-ai/langmem](https://github.com/langchain-ai/langmem) – profile vs collection память, hot-path и background memory manager, консолидация и разрешение противоречий в памяти.
- [Long-term Memory in LLM Applications — концепции LangMem](https://langchain-ai.github.io/langmem/concepts/conceptual_guide/) – очень хороший обзор о том, когда какая форма памяти уместна.
- [Memory overview — LangChain docs](https://docs.langchain.com/oss/python/concepts/memory) – более подробно как устроены основные компоненты памяти в LangChain: `Store`, namespaces, cross-thread память.
- [Awesome-Memory-for-Agents](https://github.com/TsinghuaC3I/Awesome-Memory-for-Agents) – подробный и детальный справочник по reflection, consolidation, forgetting и self-evolving памяти.